In [ ]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import pickle

from utils.graph import create_graph, add_or_update_edge, update_node_cooccurence_attr, draw_graph

In [ ]:
timeseries_dir= r"data/MIMICIII_last48h_ts2h/timeseries"
train_dir = r"data/MIMICIII_last48h_ts2h/train"
Path(train_dir).mkdir(exist_ok=True, parents=True)

## Build Comorbidity Graph

In the comorbidity graph G_c=(N_c, E_c, r_c), the nodes N_c are disease sepcified by icd9 codes, the edges E_c are the comorbidity of diseases in the same admission, and the relationship r_c is "co-occurred width".

The weight of the edge is the number of times the two icd9 codes co-occur in the same hospital admission.

In [ ]:
# load data
demo_df = pd.read_csv(os.path.join(train_dir, "demographics.csv"), sep=',')
icd9_codes = demo_df["icd9_code"].unique()
hadm_ids = demo_df["hadm_id"].unique()

diag_df = pd.read_csv(os.path.join(timeseries_dir, "diag.csv"), sep=',') # use the level 3 icd9 code
diag_df = diag_df[diag_df['hadm_id'].isin(hadm_ids)]
diag_df = diag_df[diag_df['icd9_code'].isin(icd9_codes)]

diag_df.info()

In [ ]:
# create co-occurrence graph
G_c = create_graph(diag_df.icd9_code.unique())
hadm_ids = diag_df.hadm_id.unique()

for hadm_id in hadm_ids:
    icd9_codes = diag_df[diag_df.hadm_id == hadm_id].icd9_code
    for icd9_code1 in icd9_codes:
        for icd9_code2 in icd9_codes:
            if icd9_code1 != icd9_code2:
                G_c = add_or_update_edge(G_c, icd9_code1, icd9_code2)
                G_c = update_node_cooccurence_attr(G_c, icd9_code1, icd9_code2)

# Min Max Normalization
max_weight = max([d['weight'] for _, _, d in G_c.edges(data=True)])
min_weight = min([d['weight'] for _, _, d in G_c.edges(data=True)])
for u, v, d in G_c.edges(data=True):
    G_c[u][v]['weight'] = (G_c[u][v]['weight'] - min_weight) / (max_weight - min_weight)

# # save graph
with open(os.path.join(train_dir, "comorbidity_graph.pkl"), 'wb') as f:
    pickle.dump(G_c, f)

In [ ]:
with open(os.path.join(train_dir, f"comorbidity_graph.pkl"), 'rb') as f:
    G_loaded = pickle.load(f)

draw_graph(G_loaded, node_num=20)